In [81]:
from datasets import load_dataset
from tokenizers import Tokenizer
# BPE (Byte-Pair Encoding) tokenizer from model to hold the result of trainer
from tokenizers.models import BPE
# to cut long strings recomended step by huggingface
from tokenizers.pre_tokenizers import ByteLevel
# tokenizer from trainer
from tokenizers.trainers import BpeTrainer

# for test
from tokenizers.decoders import ByteLevel as ByteLevelDecoder

# helper funcitons

In [56]:
# Function to stream the text from your dataset, to avoid everything load at once in arm
def get_corpus():
    # We loop through the 'train' split of CodeXGLUE
    for i in range(0, len(code_x_glue_dataset["train"])):
        # We give the trainer BOTH the code and the docstring
        # This way it learns 'def' (Python) and 'This function' (English)
        # 'yield' hands over the string and pauses execution. When the next item is requested,
        # the previous string is dereferenced and its memory is released.
        # if we use return it will stop the loop 
        yield code_x_glue_dataset["train"][i]["code"]
        yield code_x_glue_dataset["train"][i]["docstring"]

# Load the dataSet with python filter

In [35]:
code_x_glue_dataset = load_dataset("google/code_x_glue_ct_code_to_text", "python")

# separate train/test and validation

In [36]:
train = code_x_glue_dataset['train']
validation = code_x_glue_dataset['validation']
test = code_x_glue_dataset['test']

# tokenization

In [48]:
# 1. tokenizer

# The Skeleton/shell that uses some algo and rules that will hold the result of training 
## BPE is iterative and greedy it will iterate
## it will create merge rule and keep repeating
## words like def, if will be treated as single token at the end
## UNK is there to ensure it doesn't crash on unknow
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))

In [49]:
# 2. pre_tokenizer

# if tokenizer is the shell in which we put result of trainer then on that though ByteLevel be Scissors
# "Scissors" (Pre-tokenizer) cut the long, continuous string of Python code into individual initial chunks 
# based on specific rules (like spaces, punctuation, or byte-values).
# add_prefix_space should be false to avoid space at start of everystring
tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=False)

In [59]:
# 3. trainer setting config
# we will use Optimal (25,000 - 32,000) it will make Most Python keywords and common variables become single tokens.
# when min_freq is 2 it means the characters combo has to be appear twice to merge it to single token
# special tokens
# [SOS] (Start of Sequence), kind of green signal to start writing
# [EOS] (End of Sequence)  without this model with never start thinking
# [PAD] (Padding Token) if one python funciton shorter than other fill up space with PAD
# [UNK] (safety net) if a word in not vocab_size of 30k then it put UNK
# Standard Order: UNK=0, PAD=1, SOS=2, EOS=3
trainer = BpeTrainer(vocab_size=32000, 
                     min_frequency=2, 
                     special_tokens=["[UNK]", "[PAD]", "[SOS]", "[EOS]"])

In [60]:
# 4. trainer
# we use get_corpus to iterate through line by line
tokenizer.train_from_iterator(get_corpus(), trainer=trainer)

In [67]:
print(f"Vocab size: {tokenizer.get_vocab_size()}")

Vocab size: 32000


In [69]:
tokenizer.save("tokenizer.json")

# random testing to make sure

In [64]:
# double checking if def return if like words are token
def_id = tokenizer.token_to_id("def")
return_id = tokenizer.token_to_id("return")
if_id = tokenizer.token_to_id("if")

print(f"ids: {def_id, return_id, if_id}")

ids: (293, 582, 829)


In [68]:
# Encode a simple string
output = tokenizer.encode("def my_function")

# Check the tokens list
print(f"Tokens: {output.tokens}")
print(f"IDs: {output.ids}")
# Ġ was instead of space replace in step pre_tokenizer

Tokens: ['def', 'Ġmy', '_', 'function']
IDs: [293, 2471, 66, 1701]


In [83]:
# encode and decode should match
test_string = "def add(a, b) return a + b"
# tokenizer will add Ġ instead of space
encoded = tokenizer.encode(test_string)
# decoder setup to use ByteLevelDecoder to get rid of all those like Ġ
tokenizer.decoder = ByteLevelDecoder()
decoded_text = tokenizer.decode(encoded.ids)

print(f"Original: '{test_string}'")
print(f"Decoded:  '{decoded_text}'")
print(f"Match: {test_string == decoded_text}")

Original: 'def add(a, b) return a + b'
Decoded:  'def add(a, b) return a + b'
Match: True


In [75]:
# Your IDs for the emoji
emoji_ids = [163, 203, 196, 172]

# Standard decode often adds a space between tokens
decoded_with_spaces = tokenizer.decode(emoji_ids)

# Try decoding WITHOUT adding spaces (clean string)
# This forces the bytes to touch each other and become the emoji
clean_emoji = "".join(tokenizer.decode([i]) for i in emoji_ids).replace(" ", "")

print(f"IDs: {emoji_ids}")
print(f"Raw Decode: {decoded_with_spaces}")
print(f"Corrected Emoji: {clean_emoji}")

IDs: [163, 203, 196, 172]
Raw Decode: ð Ł ĺ Ģ
Corrected Emoji: ðŁĺĢ


In [ ]:
# remove from here

In [112]:
UNK_ID = 0
PAD_ID = 1
SOS_ID = 2
EOS_ID = 3
import torch
from torch.utils.data import Dataset

In [113]:
idx=5
code = code_x_glue_dataset["train"][idx]["code"]
docstring = code_x_glue_dataset["train"][idx]["docstring"]

In [114]:
max_length =128
code_ids = tokenizer.encode(code).ids
        # -1 because we need room for SOS or EOS token
doc_ids = tokenizer.encode(docstring).ids[:max_length - 1]

In [115]:
len(code_ids)

143

In [116]:
decoder_input = [SOS_ID] + doc_ids
decoder_target = doc_ids + [EOS_ID]

In [117]:
resp = {
    "code_ids": torch.tensor(code_ids, dtype=torch.long),
    "decoder_input": torch.tensor(decoder_input, dtype=torch.long),
    "decoder_target": torch.tensor(decoder_target, dtype=torch.long),
}

In [118]:
print(resp)

{'code_ids': tensor([  293, 13642,    11,   325,    15,  6377,    32,   333,   271,   213,
          292,  7492, 10665,   576, 13642,   275,   497,  1029,    15, 16119,
         1379,    16, 22145,  1415,   599,   262,   233,  1575,  1029,    17,
          348,  1063,   275,  2083,  3362,  1510,   346,   922,    17,   277,
         2945,    29,   237,   440,  3489,    29,   410,   998,   357,  3362,
         3635,   680,   650, 10478,  2716,    67,   357,  8394,   665,  1744,
         2688,     4,   277,  6304,    29,   237,  1950,   309,  5611,  5501,
          424,   273,   680,   650, 10478,  2716,  4103,   213,   292,   213,
          259,   234,   280,   213,  1525,   259,    29,   237,   268,   259,
           17,  1678,    29,   298,   259,   234,   259,    17,  1678,   298,
          268,   341,  6377,   357,   819,    11,    72,    15,  3489,   271,
          443,  1504,   259,   298,   771,   819,    11,  3489,    15,  1063,
          271,   443,   285,   426,   244,  6377,  